# 📅 2026-09-06 (일) 개발 노트 : 운영 정합 마무리 → 저장소·문서 정리 → 포트폴리오 → 운영 자동화 완결(비밀 교체·주간 스케줄러·백필 재가동)

## 🎯 오늘의 목표 — "운영이 로컬을 따라오게 만들고, 사람 손 없이 굴러가게 한다"

- [x] 운영 DB 정합 완료 확인 (games/game_metrics/review_* 12,843, 랭킹 3보드 로컬 s7 과 동일)
- [x] 주간 파이프라인 대상 DB 를 **운영으로 고정**(`--target prod`) + 용량 게이트 + 리뷰 이력·gem 증거 단계 추가
- [x] 킬 스위치 분리 — `STOP_BACKFILL` 이 주간 1회차까지 막던 것 수정
- [x] 저장소 폴더 정리(legacy 분리) + 폴더별 README(기능·주요 코드) + 메인 README 최신화
- [x] 상세 포트폴리오 작성(13절·트러블슈팅 20건·면접 Q&A) + 바탕화면 지원 폴더 정리
- [x] 운영 Postgres 비밀번호 교체 + 주간 스케줄러 등록
- [x] 백필 재가동 (2026-03-16~06-05, ~7,500게임 예상)


## ⭐ 15. 운영이 반년간 로컬과 갈라져 있었다 — 행 수 하나로 발견, 하루 만에 정합

**되짚기**: 09-05 심야 `gem_evidence --fill --dry-run` 을 운영에 대자 `no_reviews=4,193`. 백필 8,653·리뷰·이력·증거 지수가 전부 로컬 db 에만 있었다.

**도구를 만든 이유**: `pg_dump` 복원은 운영의 사용자 데이터(users·actions·surveys)를 덮어쓸 위험이 있다. 게임 데이터는 로컬이 원본, 사용자 데이터는 운영이 원본 — **방향이 테이블마다 다르다**. 그래서 `prod_sync` 는 app_id 기준 **upsert 전용, 삭제 없음, 공통 컬럼만, dry-run 기본**으로 만들었다.

**사고와 수습**
- 첫 실행이 운영 볼륨(0.5GB)을 가득 채워 Postgres 가 WAL 을 못 써 **크래시 루프**. 볼륨 5GB 증설 → 롤백 잔해 VACUUM(305→200MB) → 재실행.
- 행 단위 실행 61분 → `execute_batch(values_plus_batch)` 묶음 실행 **85초**.
- `raw_content/raw_reasoning` 은 로컬에선 제외해도 되지만 운영 Django 모델이 NOT NULL — dry-run 은 이걸 못 잡는다. `--include-raw` 추가. 운영 최종 399MB.

**결과 검증**: 운영 API 로 랭킹 new 1위 MECCHA CHAMELEON, steady 1위 Judofuri, 취향 검색 힐링 1위 Petal by Petal 92.7(gem 7.4, xfactor 0) — 로컬 s7 스냅샷과 동일.

**규칙화(C-13)**: 운영에 1천 행 이상 쓰기 전 `db_space` 로 용량·한도를 본다. 주간 파이프라인 0단계에 게이트(70%) + Discord 알림.


## ⭐ 16. 안전장치가 자동화를 조용히 막고 있었다 — 킬 스위치 하나를 둘로

**문제**: 09-04 비용 사고 때 만든 `data/STOP_BACKFILL` 이 남아 있었다. 이 파일은 백필 루프뿐 아니라 **주간 실행 1회차까지** 막았다. 크롤은 "0건"으로 조용히 끝나고, 알림도 정상처럼 보인다.

**수정**: 의도를 둘로 나눴다.
- `STOP_PIPELINE` — 전부 멈춤(주간 포함)
- `STOP_BACKFILL` — `--loop`(백필)만 멈춤. 주간 1회차는 통과

크롤러(`steam_crawler`)에도 같은 규칙: `STOP_PIPELINE` 은 항상, `STOP_BACKFILL` 은 `--from` 이 있을 때(=백필)만.

**교훈**: 비상 정지는 **범위를 명시**해야 한다. 범위 없는 정지 파일은 사고 뒤에 잊히고, 잊힌 채로 정상 동작을 흉내낸다.


## ⭐ 17. Railway 참조 변수는 "값 갱신"과 "프로세스 반영"이 다른 사건이다

**배경**: 09-05 심야에 운영 DB URL 을 채팅으로 주고받아 비밀번호가 노출됐다. 오늘 교체.

**절차**: `ALTER USER postgres WITH PASSWORD` → Railway pgvector 변수(`POSTGRES_PASSWORD`/`PGPASSWORD`) → 로컬 `.env PROD_DATABASE_URL` → `up -d batch` → 접속 확인(`select count(*) from games` = 12,848).

**막힌 곳**: django/fastapi 의 `DB_PASSWORD` 는 `${{Postgres.PGPASSWORD}}` **참조**라 값은 자동으로 바뀌었는데, 운영 랭킹이 500 이었다.
- `/health` → `{"status":"healthy"}` — **정적 응답이라 DB 를 안 건드린다**. 살아 있다는 증거가 아니었다.
- `/games/ranking?type=steady|new` → 둘 다 500.

**원인**: 참조 변수는 값이 갱신돼도 **이미 돌고 있는 컨테이너의 환경변수는 옛 값**이다. 두 서비스를 Redeploy 하니 즉시 정상(new 1위 MECCHA CHAMELEON).

**규칙화(C-14)**: 비밀 교체의 마지막 단계는 항상 **의존 서비스 재배포 + DB 를 실제로 읽는 엔드포인트로 확인**. 헬스체크는 무엇을 확인하는지 알고 써야 한다.
그리고 운영 비밀은 채팅에 요구하지 않는다 — `.env` 에 넣게 하고 변수명(`$PROD_DB`)으로만 다룬다.


## ⭐ 18. `docker compose up -d` 가 돌던 백필을 죽였다 — 순서의 문제

**상황**: 백필을 `exec -d` 로 띄운 **직후** 비밀번호 교체 절차의 `docker compose up -d batch` 를 실행. `up -d` 는 컨테이너를 **재생성**하므로 그 안에서 돌던 백필 프로세스가 같이 죽었다.

**피해**: 크롤 단계(시작 30초)에서 죽어 OpenAI 배치 제출 전 — **비용 0**. 오늘 날짜 `batch_tasks_*` 없음으로 확인.

**곁가지로 드러난 것**: 로그를 보려는데 `logs/` 가 batch 컨테이너에 마운트되어 있지 않았다. `weekly_pipeline.log` 는 컨테이너 안에만 쌓이고 재생성마다 사라지고 있었다. compose 에 `./logs:/app/logs` 추가 — 이제 호스트에서 `tail -f logs/weekly_pipeline.log`.

**규칙화**: **환경/비밀 변경 → `up -d` → 장기 작업 시작**. 할 일을 목록으로 만들 때도 이 순서로 배열한다. 남겨야 할 로그 디렉터리는 volumes 에 마운트한다.


## ⭐ 19. PowerShell 5.1 이 스크립트를 못 읽은 이유는 문법이 아니라 인코딩이었다

**증상**: `setup_weekly_task.ps1` 실행 시 `MissingEndCurlyBrace` — 20행 `if (...) {` 에서 닫는 중괄호가 없다는 파서 오류. 파일에는 멀쩡히 있다.

**원인**: 파일이 **BOM 없는 UTF-8**. Windows PowerShell 5.1 은 BOM 이 없으면 시스템 ANSI(CP949)로 읽는다. 한글 주석 바이트가 깨지면서 그 줄이 문자열/블록 경계를 흐트러뜨렸고, 파서는 엉뚱한 곳에서 실패했다.

**해결**: **BOM + CRLF** 로 저장. 내용은 한 글자도 안 고쳤다.

**등록 결과**: `HiddenGem_Weekly_Ingest` — 매주 월 03:30, 실행 제한 26h(OpenAI Batch 최대 24h 대기 감안), `StartWhenAvailable`(PC 꺼져 있었으면 켤 때 실행), 재시도 2회/30분.
전제: Docker Desktop 로그인 시 자동 시작 + batch 컨테이너 상시 기동.

**교훈**: "문법 오류"가 뜨면 파일이 **어떤 인코딩으로 읽히는지**부터 본다. 한글 주석이 있는 스크립트는 특히.


## 🗂 20. 저장소·포트폴리오 정리 — 남이 열었을 때 5분 안에 파악되게

- **legacy 분리**: 안 쓰는 v1 산출물(`auto_batch_sender`, `batch_merger`, `split_batch`, `db_updator`, v6 스크립트 6개, 파이프라인 v1 10개)을 `legacy/` 로 이동. 삭제가 아니라 이동 — 이력은 남긴다.
- **폴더별 README**: `embeddings/`, `fastapi_app/`, `scripts/`, `docs/`, `data/`, `frontend/`, `legacy/` 각각에 역할·주요 파일·핵심 코드 위치. 메인 README 도 최신 아키텍처로 교체.
- **포트폴리오**: `docs/portfolio_2026-09-06.md` 13절(문제정의·아키텍처·증류 데이터·v6→v7 엔진·측정 문화·운영 사고·트러블슈팅 20건·의사결정 표·AI 협업·면접 Q&A 10·수치·직군별 가이드).
  바탕화면 `포트폴리오/` 에 지원용 인덱스 구성(핵심요약 → 프로젝트별 폴더 → 면접준비), 세 프로젝트(Hidden Gem / ON Safe / 임베디드 검증)를 잇는 내러티브 초안 포함.


## 📋 다음 할 일 (09-07)

**백필 (돌아가는 중)**
- ⬜ `tail -f logs/weekly_pipeline.log` 로 회차 진행 확인 — 예상 ~7,500게임 / $30~40
- ⬜ 완료 후 `usage_report` 로 **대시보드 대조** (추정 아님), 활성/게이트 미달 비율 확인
- ⬜ 신작 리그·랭킹이 새 데이터로 어떻게 바뀌는지 스냅샷 1회

**측정**
- ⬜ s8 스냅샷: 시맨틱 3개 시나리오가 s7 과 10/10·0 변화여야 한다 (qa 캐시 검증)
- ⬜ 프롬프트 v2(앵커 분리 + 축 추가) — 전체 게임 균일 통과가 전제라 백필 후에 판단

**운영 위생**
- ⬜ `/ops/cache*` 인증 출처 확인("Authentication required" 가 코드에 없는 문구)
- ⬜ `embedding_backup_20260703`(33MB) 삭제 판단
- ⬜ Django `raw_*` `null=True` 마이그레이션 → 운영에서 raw 제거(행당 30~40% 절감)

**제품**
- ⬜ R-4 노출 게이트 재검토 / Vibe 보조축 검증 / 홈 3섹션 / 상세 페이지 배지
- ⬜ `test_core` cost_guard 4건(기존 실패) 정리
